# 02 — Start Here: Eval + Decisione Finale Fiorell.IA

Questo notebook serve per valutare l’adapter LoRA di Fiorell.IA e produrre una decisione finale GO / NO-GO.

Esegui le celle dall’alto verso il basso. Le istruzioni sono pensate per utenti non tecnici.

## Cosa produce

- `eval_adapter_scored.jsonl`
- `comparison.csv`
- `metrics_summary.json`
- `final_verdict.md`


In [ ]:
# 00 - Fiorell.IA Drive-first bootstrap
from pathlib import Path
import urllib.request

BOOTSTRAP_REL = "fiorellia_colab_drive_bootstrap.py"
DRIVE_REPO_ROOT = Path("/content/drive/MyDrive/regulatory-insight-engine")
MAC_DRIVE_REPO_ROOT = Path("/Users/itsgennymac/Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/regulatory-insight-engine")
BOOTSTRAP_URL = "https://raw.githubusercontent.com/TheGenesisAIStory/regulatory-insight-engine/main/fiorellia_colab_drive_bootstrap.py"

bootstrap_path = (DRIVE_REPO_ROOT if Path("/content").exists() else MAC_DRIVE_REPO_ROOT) / BOOTSTRAP_REL
bootstrap_path.parent.mkdir(parents=True, exist_ok=True)
if not bootstrap_path.exists() or "drive_first_bootstrap" not in bootstrap_path.read_text(encoding="utf-8", errors="ignore"):
    bootstrap_path.write_text(urllib.request.urlopen(BOOTSTRAP_URL).read().decode("utf-8"), encoding="utf-8")

exec(bootstrap_path.read_text(encoding="utf-8"), globals())


## 0 — Configurazione semplice

Controlla solo i path principali. Se hai usato il notebook training standard, il path dello ZIP è già corretto.

In [ ]:
from pathlib import Path
import sys, json, csv, subprocess

RUN_ENV = 'colab'  # colab oppure local
REPO_URL = 'https://github.com/TheGenesisAIStory/regulatory-insight-engine.git'
REPO_ROOT = Path('/content/regulatory-insight-engine') if RUN_ENV == 'colab' else Path('/Users/itsgennymac/GitHub/regulatory-insight-engine')
EVAL_ROOT = Path('/content/drive/MyDrive/fiorellia-runs/final_delivery_latest') if RUN_ENV == 'colab' else REPO_ROOT / 'artifacts/fiorellia/final_release'

ADAPTER_CANDIDATES = [
    EVAL_ROOT / 'fiorellia_behavior_20260421_clean.zip',
    Path('/content/drive/MyDrive/fiorellia-runs/fiorellia_behavior_20260421.zip') if RUN_ENV == 'colab' else REPO_ROOT / 'artifacts/fiorellia/final_release/fiorellia_behavior_20260421_clean.zip',
    Path('/content/drive/MyDrive/fiorellia/artifacts/fiorellia_lora_adapter.zip') if RUN_ENV == 'colab' else REPO_ROOT / 'artifacts/fiorellia/final_release/fiorellia_behavior_20260421_clean.zip',
]
ADAPTER_ZIP = next((p for p in ADAPTER_CANDIDATES if p.exists()), ADAPTER_CANDIDATES[0])

SYSTEM_PROMPT_PATH = REPO_ROOT / 'fiorellia/prompts/system_prompt.md'
EVAL_SET_CANDIDATES = [
    REPO_ROOT / 'fiorellia/eval/eval_set.jsonl',
    REPO_ROOT / 'fiorellia/eval/eval_set_v0.jsonl',
]
BASELINE_CANDIDATES = [
    REPO_ROOT / 'fiorellia/eval/baseline.jsonl',
    REPO_ROOT / 'fiorellia/eval/prompt_harness_baseline_20260421.jsonl',
]

EVAL_OUTPUT_JSONL = EVAL_ROOT / 'reports/adapter_eval.jsonl'
SCORED_JSONL = EVAL_ROOT / 'adapter_eval_scored.jsonl'
COMPARISON_CSV = EVAL_ROOT / 'comparison_notebook_check.csv'
METRICS_JSON = EVAL_ROOT / 'metrics_summary.json'
FINAL_VERDICT_MD = EVAL_ROOT / 'final_verdict.md'

print('Repository:', REPO_ROOT)
print('Adapter ZIP:', ADAPTER_ZIP)
print('Output eval:', EVAL_ROOT)


## 1 — Preparazione Colab

Monta Google Drive e clona il repository se serve.

In [ ]:
if RUN_ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

EVAL_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT))
print('Ambiente eval pronto.')

## 2 — Installazione dipendenze

Esegui una sola volta. Se Colab chiede riavvio, riavvia e riparti dalla sezione 0.

In [ ]:
%pip install -q \
  "transformers>=4.45,<4.52" \
  "datasets>=2.20,<3.0" \
  "accelerate>=0.33,<1.0" \
  "peft>=0.12,<0.16" \
  "trl>=0.9,<0.13" \
  "bitsandbytes>=0.43,<0.46" \
  "safetensors>=0.4" \
  "pyyaml>=6.0"
print('Dipendenze installate.')

## 3 — Controlli preliminari eval

Questa sezione verifica adapter ZIP, prompt, eval set e baseline se disponibile.

In [ ]:
from fiorellia.training.fiorellia_colab_pipeline import (
    check_cuda, validate_adapter_zip, require_file, write_json, read_jsonl, score_eval_rows, write_jsonl, write_csv, write_final_verdict
)

gpu = check_cuda(require_gpu=True)
validate_adapter_zip(ADAPTER_ZIP)

eval_set = next((p for p in EVAL_SET_CANDIDATES if p.exists()), None)
if eval_set is None:
    raise FileNotFoundError('Eval set non trovato. Controlla fiorellia/eval/ o backend/eval/.')

baseline = next((p for p in BASELINE_CANDIDATES if p.exists()), None)
system_prompt_exists = SYSTEM_PROMPT_PATH.exists()

preflight = {
    'gpu': gpu,
    'adapter_zip': str(ADAPTER_ZIP),
    'eval_set': str(eval_set),
    'baseline': str(baseline) if baseline else None,
    'system_prompt': str(SYSTEM_PROMPT_PATH) if system_prompt_exists else None,
}
write_json(preflight, EVAL_ROOT / 'preflight_eval.json')
print(json.dumps(preflight, indent=2, ensure_ascii=False))
print('Preflight eval completato.')

## 4 — Esecuzione harness adapter

Questa cella esegue lo script finale Colab/Drive. Produce output reali dell'adapter prima del calcolo metriche.


In [ ]:
if not ADAPTER_ZIP.exists():
    raise FileNotFoundError(f'Adapter ZIP non trovato: {ADAPTER_ZIP}')

cmd = [
    sys.executable,
    str(REPO_ROOT / 'fiorellia/eval/colab_drive_final_eval.py'),
    '--repo-root', str(REPO_ROOT),
    '--artifact-dir', str(EVAL_ROOT),
    '--adapter-zip', str(ADAPTER_ZIP),
]
print('Eseguo:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Output adapter:', EVAL_OUTPUT_JSONL)


## 5 — Calcolo metriche

Questa sezione calcola metriche robuste e segnala se ci sono subset vuoti che causano NaN.

In [ ]:
if not EVAL_OUTPUT_JSONL.exists():
    raise FileNotFoundError(f'Manca output eval: {EVAL_OUTPUT_JSONL}')

rows = read_jsonl(EVAL_OUTPUT_JSONL)
scored_rows, raw_metrics = score_eval_rows(rows)
release_metric_names = ['in_scope_grounded', 'unsupported_abstention', 'out_of_scope_refusal', 'italian_style']
metrics = {name: float(raw_metrics[name]) for name in release_metric_names if raw_metrics.get(name) is not None}
missing = [name for name in release_metric_names if name not in metrics]
if missing:
    raise RuntimeError(f'Metriche mancanti: {missing}')

write_jsonl(scored_rows, SCORED_JSONL)
write_json(metrics, METRICS_JSON)
write_json({'raw_metrics': raw_metrics}, EVAL_ROOT / 'eval_diagnostics_notebook.json')

print(json.dumps(metrics, indent=2, ensure_ascii=False))
print('Scored JSONL:', SCORED_JSONL)
print('Metrics JSON:', METRICS_JSON)


## 6 — CSV comparativo

Crea un CSV leggibile con caso, output e flag principali.

In [ ]:
comparison_rows = []
for i, row in enumerate(scored_rows, start=1):
    comparison_rows.append({
        'row_id': i,
        'case_type': row.get('_case_norm'),
        'pred_is_abstention': row.get('pred_is_abstention'),
        'pred_is_grounded': row.get('pred_is_grounded'),
        'pred_italian_style': row.get('pred_italian_style'),
        'output': row.get('_output_norm'),
    })
write_csv(comparison_rows, COMPARISON_CSV)
print('Notebook check CSV:', COMPARISON_CSV)
print('Baseline-vs-adapter CSV:', EVAL_ROOT / 'comparison.csv')


## 7 — Decisione finale GO / NO-GO

La decisione è GO solo se tutte le soglie sono rispettate. Se una metrica è mancante, il risultato è NO-GO.

In [ ]:
thresholds = {
    'in_scope_grounded': 0.80,
    'unsupported_abstention': 0.90,
    'out_of_scope_refusal': 0.95,
    'italian_style': 0.80,
}

def pass_metric(name):
    value = metrics.get(name)
    return value is not None and value >= thresholds[name]

checks = {name: pass_metric(name) for name in thresholds}
checks['adapter_zip_valid'] = True
checks['eval_completed'] = True

verdict = 'GO' if all(checks.values()) else 'NO-GO'
final_summary = {
    'verdict': verdict,
    'metrics': metrics,
    'thresholds': thresholds,
    'checks': checks,
    'scored_jsonl': str(SCORED_JSONL),
    'comparison_csv': str(EVAL_ROOT / 'comparison.csv'),
    'notebook_check_csv': str(COMPARISON_CSV),
}
write_json(final_summary, EVAL_ROOT / 'final_summary.json')
write_final_verdict(FINAL_VERDICT_MD, verdict, metrics)

print('CONCLUSIONE FINALE:', verdict)
print(json.dumps(final_summary, indent=2, ensure_ascii=False))
print('Final verdict markdown:', FINAL_VERDICT_MD)

if verdict != 'GO':
    print('Suggerimento: se italian_style o unsupported_abstention sono bassi, usa fiorellia/training/fiorellia_nogo_recovery_runbook.ipynb')
